# MedBoard — Train UUEKAN (Edge-Enhanced KAN) Segmentation Model

**Run this notebook in Google Colab (Free T4 GPU) or locally in VS Code.**

### What is UUEKAN?
UUEKAN (*Biomedical Signal Processing and Control*, Elsevier 2026) is an advanced medical image segmentation architecture that replaces standard MLPs with **learnable B-spline Kolmogorov-Arnold Networks (KAN)** and equips skip connections with **Uncertainty-guided Magnitude-Aware Linear Attention (U-MALA)**.

### What this notebook does:
1. **Environment Setup**: Auto-detects Colab vs Local, mounts Google Drive, copies BRISC to local SSD, installs requirements.
2. **GPU Verification**: Checks VRAM (optimized for 15GB Colab T4).
3. **Dataset Preparation**: Loads BRISC at **512×512** resolution with batch size 4.
4. **Build Architecture**: Builds UUEKAN (~8.89M parameters) with auxiliary heads.
5. **Train**: Runs training with **Gradient Accumulation (3 steps)** $\rightarrow$ **Effective Batch Size = 12** matching the paper.
6. **Loss & Metric Curves**: Plots segmentation loss, boundary loss, and Dice score.
7. **Qualitative Diagnostic**: Visualizes side-by-side **[Raw MRI | Ground Truth | Prediction | Boundary Map | Uncertainty Heatmap]**.

## Cell 1 — Environment Setup (Run Every Session)

In [ ]:
import shutil, os, sys

# ── Detect environment ───────────────────────────────────────────────
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Running in: {"Google Colab" if IS_COLAB else "Local (VS Code)"}')

if IS_COLAB:
    from google.colab import drive

    # 1. Mount Google Drive
    drive.mount('/content/drive')

    # 2. Clone or update the MedBoard repo from GitHub
    if not os.path.exists('/content/MedBoard'):
        !git clone https://github.com/shivanshu0055/Medical-Image.git /content/MedBoard
    else:
        !git -C /content/MedBoard pull

    # 3. Copy dataset from Drive to local Colab SSD (fast I/O during training)
    if not os.path.exists('/content/data'):
        print('Copying BRISC dataset to local SSD (~30 sec)...')
        shutil.copytree('/content/drive/MyDrive/medboard/data/brisc2025', '/content/data')
        print('Dataset copied to /content/data.')
    else:
        print('Dataset already available on local SSD.')

    # 4. Install dependencies
    !pip install -q -r /content/MedBoard/requirements_colab.txt

    # 5. Set Python path
    if '/content/MedBoard' not in sys.path:
        sys.path.insert(0, '/content/MedBoard')

    # Paths
    DATA_ROOT   = '/content/data'
    WEIGHTS_DIR = '/content/drive/MyDrive/medboard/weights'
    CONFIG_PATH = '/content/MedBoard/configs/config_uuekan.yaml'

else:
    # ── Local / VS Code paths ────────────────────────────────────────
    REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.exists(os.path.join(os.getcwd(), '..', 'modules')) else os.getcwd()
    DATA_ROOT   = os.path.join(REPO_ROOT, 'data', 'raw')
    WEIGHTS_DIR = os.path.join(REPO_ROOT, 'weights')
    CONFIG_PATH = os.path.join(REPO_ROOT, 'configs', 'config_uuekan.yaml')

    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

os.makedirs(WEIGHTS_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, 'uuekan_best.pth')
FINAL_PATH      = os.path.join(WEIGHTS_DIR, 'uuekan_brisc.pth')

print(f'\nSetup complete!')
print(f'  Data Path       -> {DATA_ROOT}')
print(f'  Weights Path    -> {WEIGHTS_DIR}')
print(f'  Best Checkpoint -> {CHECKPOINT_PATH}')

## Cell 2 — Verify GPU & Memory Status

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU Device      : {device_name}')
    print(f'Total VRAM      : {vram_gb:.2f} GB')
    if vram_gb >= 14.0:
        print('Excellent! High VRAM detected (Colab T4 / A100). Ready for 512x512 resolution.')
    else:
        print(f'Note: VRAM is {vram_gb:.1f}GB. Gradient accumulation enabled to prevent OOM.')
else:
    print('WARNING: Running on CPU. Training will be slow. Switch to a GPU runtime if on Colab!')

## Cell 3 — Load BRISC Dataset at 512×512 Resolution

In [ ]:
from modules.uuekan.dataset import get_uuekan_dataloaders

train_images = f'{DATA_ROOT}/segmentation_task/train/images'
train_masks  = f'{DATA_ROOT}/segmentation_task/train/masks'
test_images  = f'{DATA_ROOT}/segmentation_task/test/images'
test_masks   = f'{DATA_ROOT}/segmentation_task/test/masks'

IMG_SIZE   = 512
BATCH_SIZE = 4   # 4 samples * 3 grad_accum_steps = 12 effective batch size

train_loader, val_loader, test_loader = get_uuekan_dataloaders(
    train_images_dir = train_images,
    train_masks_dir  = train_masks,
    test_images_dir  = test_images,
    test_masks_dir   = test_masks,
    img_size   = IMG_SIZE,
    batch_size = BATCH_SIZE,
    val_split  = 0.1,
    num_workers= 2,
    seed       = 42,
)

# Check one sample
sample_img, sample_mask, sample_name = next(iter(train_loader))
print(f'Sample batch images : {sample_img.shape}  dtype: {sample_img.dtype}  range: [{sample_img.min():.2f}, {sample_img.max():.2f}]')
print(f'Sample batch masks  : {sample_mask.shape} dtype: {sample_mask.dtype} range: [{sample_mask.min():.2f}, {sample_mask.max():.2f}]')

## Cell 4 — Build UUEKAN Architecture with Auxiliary Heads

In [ ]:
import yaml
from modules.uuekan.uuekan_model import build_uuekan

# Load config
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, 'r') as f:
        cfg = yaml.safe_load(f)
    print(f'Loaded config from: {CONFIG_PATH}')
else:
    cfg = {'model': {'img_size': IMG_SIZE, 'use_uncertainty': True}}
    print('Using default in-memory config.')

model = build_uuekan(cfg)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n[UUEKAN Architecture Summary]')
print(f'  Total Parameters     : {total_params:,} ({total_params/1e6:.2f}M)')
print(f'  Trainable Parameters : {trainable_params:,} ({trainable_params/1e6:.2f}M)')

# Quick sanity check forward pass with auxiliary outputs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
with torch.no_grad():
    main_out, aux_outs, unc_maps = model(dummy_input, auxiliary=True)

print(f'Sanity Check Passed:')
print(f'  Main Output Shape    : {tuple(main_out.shape)}')
print(f'  Auxiliary Heads      : {len(aux_outs)} heads -> shapes: {[tuple(a.shape) for a in aux_outs]}')
print(f'  Uncertainty Maps     : {len(unc_maps)} stages -> shapes: {[tuple(u.shape) for u in unc_maps]}')

## Cell 5 — Train UUEKAN Model

We train using:
- **Tri-Partite Loss**: $L_{total} = L_{seg} + 0.2 \cdot L_{boundary} + 0.1 \cdot L_{uncertainty}$
- **Effective Batch Size = 12**: Physical batch size 4 with 3 gradient accumulation steps.
- **Cosine Annealing LR**: Initial $1 \times 10^{-4}$ decaying to $1 \times 10^{-5}$.

In [ ]:
from modules.uuekan.trainer import train_uuekan

EPOCHS              = 50
LEARNING_RATE       = 1e-4
MIN_LR              = 1e-5
GRAD_ACCUM_STEPS    = 3      # 4 * 3 = 12 effective batch size
PATIENCE            = 10     # Early stopping
RESUME              = True   # Set to True to pick up from checkpoint, False to restart

history = train_uuekan(
    model              = model,
    train_loader       = train_loader,
    val_loader         = val_loader,
    epochs             = EPOCHS,
    learning_rate      = LEARNING_RATE,
    min_lr             = MIN_LR,
    grad_accum_steps   = GRAD_ACCUM_STEPS,
    patience           = PATIENCE,
    checkpoint_path    = CHECKPOINT_PATH,
    device_str         = 'cuda' if torch.cuda.is_available() else 'cpu',
    resume             = RESUME,
)

# Save final weights
torch.save(model.state_dict(), FINAL_PATH)
print(f'\n[Done] Training complete! Best checkpoint: {CHECKPOINT_PATH}')


## Cell 6 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Total Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', lw=2)
axes[0].plot(epochs_range, history['val_loss'], 'r--', label='Val Loss', lw=2)
axes[0].set_title('Composite Loss (Seg + Boundary + Uncertainty)', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Dice Similarity Coefficient
axes[1].plot(epochs_range, history['train_dice'], 'b-', label='Train Dice', lw=2)
axes[1].plot(epochs_range, history['val_dice'], 'r--', label='Val Dice', lw=2)
axes[1].set_title('Dice Similarity Coefficient (DSC)', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Intersection over Union (IoU)
axes[2].plot(epochs_range, history['train_iou'], 'b-', label='Train IoU', lw=2)
axes[2].plot(epochs_range, history['val_iou'], 'r--', label='Val IoU', lw=2)
axes[2].set_title('Intersection over Union (IoU)', fontsize=12)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('IoU')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 7 — Qualitative Visualizations: Predictions, Boundaries & Uncertainty Heatmaps

In [ ]:
import numpy as np
import torch.nn.functional as F

model.eval()
test_images_batch, test_masks_batch, filenames = next(iter(test_loader))
test_images_batch = test_images_batch.to(device)

with torch.no_grad():
    main_pred, aux_preds, unc_maps = model(test_images_batch, auxiliary=True)
    pred_probs = torch.sigmoid(main_pred).cpu()

# Display 3 samples from test set
num_samples = min(3, test_images_batch.shape[0])

fig, axes = plt.subplots(num_samples, 5, figsize=(20, 4 * num_samples))
if num_samples == 1:
    axes = np.expand_dims(axes, 0)

for i in range(num_samples):
    # 1. Raw MRI
    raw_mri = test_images_batch[i].cpu().permute(1, 2, 0).numpy()
    axes[i, 0].imshow(raw_mri)
    axes[i, 0].set_title(f'Raw MRI\n{filenames[i]}')
    axes[i, 0].axis('off')

    # 2. Ground Truth Mask
    gt_mask = test_masks_batch[i, 0].cpu().numpy()
    axes[i, 1].imshow(gt_mask, cmap='gray')
    axes[i, 1].set_title('Ground Truth Mask')
    axes[i, 1].axis('off')

    # 3. UUEKAN Predicted Mask
    pred_binary = (pred_probs[i, 0].numpy() > 0.5).astype(np.float32)
    axes[i, 2].imshow(pred_binary, cmap='gray')
    axes[i, 2].set_title('UUEKAN Prediction')
    axes[i, 2].axis('off')

    # 4. Predicted Probability Map
    prob_map = pred_probs[i, 0].numpy()
    axes[i, 3].imshow(prob_map, cmap='magma')
    axes[i, 3].set_title('Probability Map')
    axes[i, 3].axis('off')

    # 5. Uncertainty Heatmap (Highest uncertainty along ambiguous boundaries)
    if len(unc_maps) > 0:
        # Resize shallowest uncertainty map to full resolution
        u_full = F.interpolate(unc_maps[-1][i:i+1], size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        u_np = u_full[0, 0].cpu().numpy()
    else:
        # Fallback to decision boundary distance
        u_np = 0.5 - np.abs(prob_map - 0.5)
    
    im = axes[i, 4].imshow(u_np, cmap='inferno')
    axes[i, 4].set_title('Boundary Uncertainty Map')
    axes[i, 4].axis('off')

plt.tight_layout()
plt.show()

## Cell 8 — Full Quantitative Evaluation on Test Set & Benchmark Comparison

Computes test metrics (Dice, IoU, Precision, Recall, Loss) across all test samples, compares directly with the **Base U-Net (87.01% Dice)**, and saves results to `metrics_uuekan.json`.

In [ ]:
from modules.uuekan.trainer import evaluate_test_set

# Metrics output path
if IS_COLAB:
    METRICS_JSON_PATH = '/content/MedBoard/metrics/segmentation/metrics_uuekan.json'
else:
    METRICS_JSON_PATH = os.path.join(REPO_ROOT, 'metrics', 'segmentation', 'metrics_uuekan.json')

# Run comprehensive test set evaluation
test_results = evaluate_test_set(
    model          = model,
    test_loader    = test_loader,
    device         = device,
    save_json_path = METRICS_JSON_PATH,
    base_dice      = 0.8701,  # Base U-Net Dice score on BRISC test set
    base_iou       = 0.8021,  # Base U-Net IoU score on BRISC test set
)

# If running on Colab, save a backup copy to Google Drive
if IS_COLAB:
    import json
    drive_backup = '/content/drive/MyDrive/medboard/metrics_uuekan.json'
    os.makedirs(os.path.dirname(drive_backup), exist_ok=True)
    with open(drive_backup, 'w') as f:
        json.dump(test_results, f, indent=2)
    print(f'Test metrics backup also saved to Google Drive: {drive_backup}')
